In [0]:
client_id     = dbutils.secrets.get(scope="kv-scope", key="sp-client-id")
tenant_id     = dbutils.secrets.get(scope="kv-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="sp-client-secret")

storage_account = "azurelabadls225"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

RAW_PATH       = f"abfss://raw@{storage_account}.dfs.core.windows.net"
PROCESSED_PATH = f"abfss://processed@{storage_account}.dfs.core.windows.net"
CURATED_PATH   = f"abfss://curated@{storage_account}.dfs.core.windows.net"

# Enable Adaptive Query Execution — Spark auto-optimizes skew
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("Setup complete")

Setup complete


In [0]:
df_silver = spark.read.format("delta").load(f"{PROCESSED_PATH}/delta/silver_yellow_taxi")
print(f"Silver rows: {df_silver.count():,}")

Silver rows: 35,574,985


In [0]:
from pyspark.sql.functions import count, col

# Show trip distribution by borough — this proves Manhattan skew
df_silver.groupBy("pickup_borough") \
    .agg(count("*").alias("trip_count")) \
    .orderBy(col("trip_count").desc()) \
    .show()

+--------------+----------+
|pickup_borough|trip_count|
+--------------+----------+
|     Manhattan|  31518365|
|        Queens|   3504125|
|       Unknown|    313417|
|      Brooklyn|    182142|
|         Bronx|     47822|
|           N/A|      6342|
| Staten Island|      1744|
|           EWR|      1028|
+--------------+----------+



In [0]:
from pyspark.sql.functions import (sum, avg, count, round, 
                                    col, lit, rand, floor)

# SKEW HANDLING WITH SALTING
# Manhattan has 80% of trips — one partition gets overloaded
# Fix: add random salt to spread Manhattan across multiple partitions

SALT_BUCKETS = 4  # split each borough into 4 sub-partitions

# Step 1 — Add salt key to Silver data
df_salted = df_silver.withColumn("salt", (floor(rand() * SALT_BUCKETS)).cast("int"))

# Step 2 — First aggregation WITH salt (partial results per salt bucket)
df_partial = (df_salted
    .groupBy("pickup_borough", "pickup_zone_name", "salt")
    .agg(
        count("*").alias("trip_count"),
        sum("fare_amount").alias("total_fare"),
        sum("tip_amount").alias("total_tips"),
        sum("total_amount").alias("total_revenue"),
        avg("trip_distance").alias("avg_distance")
    )
)

# Step 3 — Second aggregation WITHOUT salt (combine partial results)
df_revenue_by_zone = (df_partial
    .groupBy("pickup_borough", "pickup_zone_name")
    .agg(
        sum("trip_count").alias("trip_count"),
        round(sum("total_fare"), 2).alias("total_fare"),
        round(sum("total_tips"), 2).alias("total_tips"),
        round(sum("total_revenue"), 2).alias("total_revenue"),
        round(avg("avg_distance"), 2).alias("avg_distance")
    )
    .orderBy(col("total_revenue").desc())
)

print("✅ Revenue by zone calculated with skew handling")
df_revenue_by_zone.show(10)

✅ Revenue by zone calculated with skew handling
+--------------+--------------------+----------+--------------+-------------+--------------+------------+
|pickup_borough|    pickup_zone_name|trip_count|    total_fare|   total_tips| total_revenue|avg_distance|
+--------------+--------------------+----------+--------------+-------------+--------------+------------+
|        Queens|         JFK Airport|   1887934|1.2027694234E8|1.691888179E7|1.5359675618E8|       16.23|
|        Queens|   LaGuardia Airport|   1262221| 5.496558409E7|1.109048584E7| 8.471150734E7|        9.97|
|     Manhattan|      Midtown Center|   1674830| 2.794437151E7|   5489669.35| 4.240741703E7|        2.57|
|     Manhattan|Upper East Side S...|   1701175| 2.233806721E7|   4587910.03| 3.517444449E7|        1.81|
|     Manhattan|Times Sq/Theatre ...|   1196974| 2.339523157E7|   4223083.81| 3.450116997E7|        3.34|
|     Manhattan|Upper East Side N...|   1500384|  2.04258131E7|   4193541.47|   3.1827268E7|        2.03

In [0]:
from pyspark.sql.functions import sum, avg, count, round, col

df_hourly_demand = (df_silver
    .groupBy("pickup_hour", "pickup_day")
    .agg(
        count("*").alias("trip_count"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("trip_duration_mins"), 2).alias("avg_duration_mins"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy("pickup_hour", "pickup_day")
)

print("✅ Hourly demand pattern calculated")
df_hourly_demand.show(10)

✅ Hourly demand pattern calculated
+-----------+----------+----------+--------+-----------------+-------------+
|pickup_hour|pickup_day|trip_count|avg_fare|avg_duration_mins|total_revenue|
+-----------+----------+----------+--------+-----------------+-------------+
|          0|         1|    253049|   17.89|            13.97|   6692583.77|
|          0|         2|     74081|   27.07|            14.99|   2813234.48|
|          0|         3|     68211|    25.9|            15.13|   2483505.92|
|          0|         4|     89826|   22.22|            14.27|   2854010.64|
|          0|         5|    110446|   20.71|            13.92|   3314418.02|
|          0|         6|    154971|   19.58|            13.89|   4448836.29|
|          0|         7|    244519|   18.09|            14.17|   6560555.15|
|          1|         1|    212619|   16.43|            12.89|    5240167.8|
|          1|         2|     38050|   25.73|            14.17|    1353972.4|
|          1|         3|     31118|   25.

In [0]:
from pyspark.sql.functions import sum, avg, count, round, col, when

df_payment = (df_silver
    .withColumn("payment_type_name",
        when(col("payment_type") == 1, "Credit Card")
        .when(col("payment_type") == 2, "Cash")
        .when(col("payment_type") == 3, "No Charge")
        .when(col("payment_type") == 4, "Dispute")
        .otherwise("Unknown"))
    .groupBy("payment_type_name")
    .agg(
        count("*").alias("trip_count"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(avg("tip_amount"), 2).alias("avg_tip"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy(col("trip_count").desc())
)

print("✅ Payment analysis complete")
df_payment.show()

✅ Payment analysis complete
+-----------------+----------+--------+-------+--------------+
|payment_type_name|trip_count|avg_fare|avg_tip| total_revenue|
+-----------------+----------+--------+-------+--------------+
|      Credit Card|  29122539|   19.69|    4.4|8.6559951017E8|
|             Cash|   6084855|   20.02|    0.0|1.5413860984E8|
|          Dispute|    242929|   20.44|   0.02|    6293932.95|
|        No Charge|    124662|   18.37|   0.01|    2940979.89|
+-----------------+----------+--------+-------+--------------+



In [0]:
gold_path = f"{CURATED_PATH}/delta/gold_nyc_taxi"

# Union all KPIs into separate Gold tables
df_revenue_by_zone.write.format("delta").mode("overwrite") \
    .save(f"{CURATED_PATH}/delta/gold_revenue_by_zone")

df_hourly_demand.write.format("delta").mode("overwrite") \
    .save(f"{CURATED_PATH}/delta/gold_hourly_demand")

df_payment.write.format("delta").mode("overwrite") \
    .save(f"{CURATED_PATH}/delta/gold_payment_analysis")

print("✅ All Gold tables written!")

✅ All Gold tables written!


In [0]:
# Z-ORDER clusters related data together in files
# Queries filtering by borough + hour will scan far fewer files

spark.sql(f"""
    OPTIMIZE delta.`{CURATED_PATH}/delta/gold_revenue_by_zone`
    ZORDER BY (pickup_borough, pickup_zone_name)
""")

spark.sql(f"""
    OPTIMIZE delta.`{CURATED_PATH}/delta/gold_hourly_demand`
    ZORDER BY (pickup_hour, pickup_day)
""")

print("✅ Z-ORDER optimization complete!")

✅ Z-ORDER optimization complete!


In [0]:
df_gold = spark.read.format("delta").load(f"{CURATED_PATH}/delta/gold_revenue_by_zone")
print(f"Gold revenue table rows: {df_gold.count()}")

df_gold_hourly = spark.read.format("delta").load(f"{CURATED_PATH}/delta/gold_hourly_demand")
print(f"Gold hourly table rows: {df_gold_hourly.count()}")

df_gold_payment = spark.read.format("delta").load(f"{CURATED_PATH}/delta/gold_payment_analysis")
print(f"Gold payment table rows: {df_gold_payment.count()}")

print("\n🎉 FULL PIPELINE COMPLETE!")
print("Bronze → Silver → Gold ✅")

Gold revenue table rows: 261
Gold hourly table rows: 168
Gold payment table rows: 4

🎉 FULL PIPELINE COMPLETE!
Bronze → Silver → Gold ✅


In [0]:
print("=" * 60)
print("🏆 NYC TAXI PIPELINE — FULL RESULTS SUMMARY")
print("=" * 60)

print("\n📊 PIPELINE STATS:")
print(f"  Bronze rows (raw):     38,310,226")
print(f"  Silver rows (cleaned): 35,574,985")
print(f"  Rows removed:           2,735,241 (7.1% dirty data)")

print("\n" + "=" * 60)
print("🥇 GOLD KPI 1: TOP 20 REVENUE ZONES")
print("=" * 60)
df_gold_rev = spark.read.format("delta").load(f"{CURATED_PATH}/delta/gold_revenue_by_zone")
df_gold_rev.orderBy(col("total_revenue").desc()).show(20, truncate=False)

print("\n" + "=" * 60)
print("🥇 GOLD KPI 2: HOURLY DEMAND PATTERN")
print("=" * 60)
df_gold_hourly = spark.read.format("delta").load(f"{CURATED_PATH}/delta/gold_hourly_demand")
df_gold_hourly.orderBy(col("trip_count").desc()).show(24, truncate=False)

print("\n" + "=" * 60)
print("🥇 GOLD KPI 3: PAYMENT TYPE ANALYSIS")
print("=" * 60)
df_gold_payment = spark.read.format("delta").load(f"{CURATED_PATH}/delta/gold_payment_analysis")
df_gold_payment.orderBy(col("trip_count").desc()).show(truncate=False)

print("\n" + "=" * 60)
print("🔍 SKEW PROOF — TRIP DISTRIBUTION BY BOROUGH")
print("=" * 60)
df_silver = spark.read.format("delta").load(f"{PROCESSED_PATH}/delta/silver_yellow_taxi")
df_silver.groupBy("pickup_borough") \
    .agg(count("*").alias("trip_count"),
         round(sum("total_amount"), 2).alias("total_revenue")) \
    .orderBy(col("trip_count").desc()) \
    .show(truncate=False)

print("\n" + "=" * 60)
print("📅 MONTHLY TRIP TRENDS")
print("=" * 60)
df_silver.groupBy("pickup_month") \
    .agg(count("*").alias("trip_count"),
         round(avg("fare_amount"), 2).alias("avg_fare"),
         round(sum("total_amount"), 2).alias("total_revenue")) \
    .orderBy("pickup_month") \
    .show(truncate=False)

print("\n🎉 END OF RESULTS")

🏆 NYC TAXI PIPELINE — FULL RESULTS SUMMARY

📊 PIPELINE STATS:
  Bronze rows (raw):     38,310,226
  Silver rows (cleaned): 35,574,985
  Rows removed:           2,735,241 (7.1% dirty data)

🥇 GOLD KPI 1: TOP 20 REVENUE ZONES
+--------------+----------------------------+----------+--------------+-------------+--------------+------------+
|pickup_borough|pickup_zone_name            |trip_count|total_fare    |total_tips   |total_revenue |avg_distance|
+--------------+----------------------------+----------+--------------+-------------+--------------+------------+
|Queens        |JFK Airport                 |1887934   |1.2027694234E8|1.691888179E7|1.5359675618E8|16.23       |
|Queens        |LaGuardia Airport           |1262221   |5.496558409E7 |1.109048584E7|8.471150734E7 |9.97        |
|Manhattan     |Midtown Center              |1674830   |2.794437151E7 |5489669.35   |4.240741703E7 |2.57        |
|Manhattan     |Upper East Side South       |1701175   |2.233806721E7 |4587910.03   |3.51744

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS gold_revenue_by_zone
USING DELTA
LOCATION '{CURATED_PATH}/delta/gold_revenue_by_zone'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS gold_hourly_demand
USING DELTA
LOCATION '{CURATED_PATH}/delta/gold_hourly_demand'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS gold_payment_analysis
USING DELTA
LOCATION '{CURATED_PATH}/delta/gold_payment_analysis'
""")

print("✅ All Gold tables registered!")

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-8116629752475034>, line 1
----> 1 spark.sql(f"""
      2 CREATE TABLE IF NOT EXISTS gold_revenue_by_zone
      3 USING DELTA
      4 LOCATION '{CURATED_PATH}/delta/gold_revenue_by_zone'
      5 """)
      7 spark.sql(f"""
      8 CREATE TABLE IF NOT EXISTS gold_hourly_demand
      9 USING DELTA
     10 LOCATION '{CURATED_PATH}/delta/gold_hourly_demand'
     11 """)
     13 spark.sql(f"""
     14 CREATE TABLE IF NOT EXISTS gold_payment_analysis
     15 USING DELTA
     16 LOCATION '{CURATED_PATH}/delta/gold_payment_analysis'
     17 """)

File /databricks/spark/python/pyspark/databricks/instrumentation/instrumentation_utils.py:217, in _wrap_function.<locals>.wrapper(*args, **kwargs)
    215 start = time.perf_counter()
    216 try:
--> 217     res = func(*args, **kwargs)
    218     logging_helper.log_event(
    219         

In [0]:
client_id     = dbutils.secrets.get(scope="kv-scope", key="sp-client-id")
tenant_id     = dbutils.secrets.get(scope="kv-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="sp-client-secret")
storage_account = "azurelabadls225"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

df1 = spark.read.format("delta").load(f"abfss://curated@{storage_account}.dfs.core.windows.net/delta/gold_revenue_by_zone")
df1.createOrReplaceTempView("gold_revenue_by_zone")

df2 = spark.read.format("delta").load(f"abfss://curated@{storage_account}.dfs.core.windows.net/delta/gold_hourly_demand")
df2.createOrReplaceTempView("gold_hourly_demand")

df3 = spark.read.format("delta").load(f"abfss://curated@{storage_account}.dfs.core.windows.net/delta/gold_payment_analysis")
df3.createOrReplaceTempView("gold_payment_analysis")

print("✅ Views created!")
spark.sql("SELECT * FROM gold_revenue_by_zone LIMIT 3").show()

✅ Views created!
+--------------+-----------------+----------+--------------+-------------+--------------+------------+
|pickup_borough| pickup_zone_name|trip_count|    total_fare|   total_tips| total_revenue|avg_distance|
+--------------+-----------------+----------+--------------+-------------+--------------+------------+
|        Queens|      JFK Airport|   1887934|1.2027694234E8|1.691888179E7|1.5359675618E8|       16.23|
|        Queens|LaGuardia Airport|   1262221| 5.496558409E7|1.109048584E7| 8.471150734E7|        9.97|
|     Manhattan|   Midtown Center|   1674830| 2.794437151E7|   5489669.35| 4.240741703E7|        2.57|
+--------------+-----------------+----------+--------------+-------------+--------------+------------+



In [0]:
from pyspark.sql.functions import count, sum, avg, round, col

client_id     = dbutils.secrets.get(scope="kv-scope", key="sp-client-id")
tenant_id     = dbutils.secrets.get(scope="kv-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="sp-client-secret")
storage_account = "azurelabadls225"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

CURATED_PATH   = f"abfss://curated@{storage_account}.dfs.core.windows.net"
PROCESSED_PATH = f"abfss://processed@{storage_account}.dfs.core.windows.net"

# Read existing Gold Delta tables
df_revenue = spark.read.format("delta").load(f"{CURATED_PATH}/delta/gold_revenue_by_zone")
df_hourly  = spark.read.format("delta").load(f"{CURATED_PATH}/delta/gold_hourly_demand")
df_payment = spark.read.format("delta").load(f"{CURATED_PATH}/delta/gold_payment_analysis")
df_silver  = spark.read.format("delta").load(f"{PROCESSED_PATH}/delta/silver_yellow_taxi")

df_monthly = df_silver.groupBy("pickup_month") \
    .agg(count("*").alias("trip_count"),
         round(avg("fare_amount"),2).alias("avg_fare"),
         round(sum("total_amount"),2).alias("total_revenue")) \
    .orderBy("pickup_month")

df_borough = df_silver.groupBy("pickup_borough") \
    .agg(count("*").alias("trip_count"),
         round(sum("total_amount"),2).alias("total_revenue")) \
    .orderBy(col("trip_count").desc())

# Write to temp folders first
temp_path = f"{CURATED_PATH}/csv_temp"
df_revenue.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{temp_path}/revenue")
df_hourly.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{temp_path}/hourly")
df_payment.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{temp_path}/payment")
df_monthly.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{temp_path}/monthly")
df_borough.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{temp_path}/borough")

# Rename part files to clean names
clean_path = f"{CURATED_PATH}/powerbi_exports"

files = {
    "revenue": "01_revenue_by_zone.csv",
    "hourly":  "02_hourly_demand.csv",
    "payment": "03_payment_analysis.csv",
    "monthly": "04_monthly_trends.csv",
    "borough": "05_borough_skew.csv",
}

for folder, clean_name in files.items():
    # Find the part file
    file_list = dbutils.fs.ls(f"{temp_path}/{folder}/")
    part_file = [f.path for f in file_list if f.name.startswith("part-")][0]
    
    # Copy with clean name
    dbutils.fs.cp(part_file, f"{clean_path}/{clean_name}")
    print(f"✅ {clean_name}")

# Cleanup temp
dbutils.fs.rm(temp_path, recurse=True)

print("\n🎉 Clean CSVs ready!")
print(f"Location: curated/powerbi_exports/")
print("\nFiles:")
for f in dbutils.fs.ls(clean_path):
    print(f"  {f.name}")

✅ 01_revenue_by_zone.csv
✅ 02_hourly_demand.csv
✅ 03_payment_analysis.csv
✅ 04_monthly_trends.csv
✅ 05_borough_skew.csv

🎉 Clean CSVs ready!
Location: curated/powerbi_exports/

Files:
  01_revenue_by_zone.csv
  02_hourly_demand.csv
  03_payment_analysis.csv
  04_monthly_trends.csv
  05_borough_skew.csv
